# NB3 — FastAPI `/search` Endpoint + Latency Benchmark

**Stack:** FastAPI + uvicorn + httpx (client). Searcher từ `app/search.py`.
Maps to slide §7 (Production Patterns) + deliverable bullets 1, 4.

> Mục tiêu: bọc `Searcher` thành REST API, đo P50/P95/P99 latency, đảm bảo
> P99 < 50 ms cho hybrid mode (rubric threshold).

In [1]:
import _setup  # noqa: F401
import statistics
import subprocess
import socket
import time
from pathlib import Path

import httpx

## 1. Khởi động API server (background)

Trong production thực tế, bạn sẽ chạy `make api` ở terminal riêng. Notebook
này khởi động uvicorn ở background subprocess và đợi `/healthz` trả ready.

In [2]:
ROOT = Path(_setup.__file__).resolve().parent.parent
with socket.socket() as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    API_PORT = port_probe.getsockname()[1]
proc = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(API_PORT), "--log-level", "warning"],
    cwd=str(ROOT),
)

# Đợi server up + warm (Searcher.from_corpus loads embeddings + indexes 1000 docs)
URL = f"http://127.0.0.1:{API_PORT}"
for _ in range(180):
    try:
        r = httpx.get(f"{URL}/healthz", timeout=2.0)
        if r.status_code == 200 and r.json().get("ready"):
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("API didn't become ready within 180s")

print(httpx.get(f"{URL}/healthz").json())

{'ready': True, 'n_docs': 1000}


## 2. Single query — kiểm tra response shape

In [3]:
r = httpx.get(f"{URL}/search", params={"q": "cloud computing tự động mở rộng", "mode": "hybrid"})
r.raise_for_status()
body = r.json()
assert {"query", "mode", "top_k", "latency_ms", "hits"} <= body.keys()
assert not body["hits"] or {"doc_id", "title", "text", "score"} <= body["hits"][0].keys()
print("Response fields:", ", ".join(body.keys()))
print(f"Sample hit fields: {', '.join(body['hits'][0].keys()) if body['hits'] else '(no hits)'}")
print(f"query={body['query']!r} mode={body['mode']} top_k={body['top_k']} latency_ms={body['latency_ms']:.1f}")
for h in body["hits"][:3]:
    print(f"  {h['doc_id']:>14}  score={h['score']:.4f}  {h['title']}")

Response fields: query, mode, top_k, latency_ms, hits
Sample hit fields: doc_id, title, text, score
query='cloud computing tự động mở rộng' mode=hybrid top_k=10 latency_ms=23.6
       cloud_016  score=0.0325  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_072  score=0.0323  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_053  score=0.0315  Điện toán đám mây: tự động mở rộng theo lưu lượng


## 3. Latency benchmark (100 queries × 3 modes)

Dùng 50 golden queries × 2 reps = 100 calls/mode. Ghi nhận latency từ
`body["latency_ms"]` (server-side, đã trừ network) HOẶC từ wall-clock httpx
(bao gồm network) — note: rubric assert P99 < 50ms áp dụng cho server-side.

Output: bảng P50/P95/P99 cho 3 mode.

In [4]:
import json

DATA = ROOT / "data"
golden = [json.loads(l) for l in (DATA / "golden_set.jsonl").open(encoding="utf-8")]

# Warm the embedding model and retrieval path before measuring server latency.
warmup_count = min(10, len(golden))
for q in golden[:warmup_count]:
    warmup_response = httpx.get(f"{URL}/search", params={"q": q["query"], "mode": "hybrid"})
    warmup_response.raise_for_status()
print(f"Warmup complete: {warmup_count} queries")


def percentile(values: list[float], p: float) -> float:
    n = len(values)
    if n == 0:
        return 0.0
    ordered = sorted(values)
    position = (n - 1) * p
    lower = int(position)
    upper = min(lower + 1, n - 1)
    fraction = position - lower
    return ordered[lower] + (ordered[upper] - ordered[lower]) * fraction


def benchmark_mode(mode: str, reps: int = 2) -> dict[str, float]:
    server_latencies: list[float] = []
    wall_latencies: list[float] = []
    for _ in range(reps):
        for q in golden:
            t0 = time.perf_counter()
            r = httpx.get(f"{URL}/search", params={"q": q["query"], "mode": mode})
            r.raise_for_status()
            wall_latencies.append((time.perf_counter() - t0) * 1000)
            server_latencies.append(r.json()["latency_ms"])
    return {
        "p50_server": percentile(server_latencies, 0.50),
        "p95_server": percentile(server_latencies, 0.95),
        "p99_server": percentile(server_latencies, 0.99),
        "p99_wall":   percentile(wall_latencies, 0.99),
    }


print(f"  {'mode':10}  {'P50':>7}  {'P95':>7}  {'P99':>7}  {'P99(wall)':>9}")
results = {}
for mode in ("keyword", "semantic", "hybrid"):
    res = benchmark_mode(mode)
    results[mode] = res
    print(f"  {mode:10}  {res['p50_server']:>5.1f}ms  {res['p95_server']:>5.1f}ms  "
          f"{res['p99_server']:>5.1f}ms  {res['p99_wall']:>7.1f}ms")

Warmup complete: 10 queries
  mode            P50      P95      P99  P99(wall)


  keyword       3.2ms    4.5ms    5.1ms     85.2ms


  semantic     11.8ms   16.3ms   19.5ms     80.1ms


  hybrid       16.8ms   24.1ms   29.1ms    140.4ms


## 4. Rubric assertion — hybrid P99 server-side < 50ms

In [5]:
hybrid_p99 = results["hybrid"]["p99_server"]
print(f"Hybrid P99 server-side: {hybrid_p99:.1f}ms")
if hybrid_p99 < 50:
    print(f"PASS — hybrid P99 < 50ms ({hybrid_p99:.1f}ms)")
else:
    print(f"WARN — hybrid P99 >= 50ms ({hybrid_p99:.1f}ms)")
    print("  Possible causes: cold cache, fastembed model not warm yet, or RRF depth=50 is too aggressive")
    print("  Check the measured server-side distribution before drawing a performance conclusion.")

Hybrid P99 server-side: 29.1ms
PASS — hybrid P99 < 50ms (29.1ms)


## 5. Cleanup — stop the API server

In [6]:
proc.terminate()
try:
    proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait(timeout=5)
print("API server stopped")

API server stopped


## Deliverable evidence

1. Output cell 2: 1 single hybrid query response with `top-3 hits`.
2. Output cell 3: latency table P50/P95/P99 for keyword/semantic/hybrid.
3. Output cell 4: hybrid P99 < 50ms PASS.

---

## Vibe-coding callout

**Delegate freely:** the FastAPI scaffolding (route definition, Pydantic
response model, lifespan handler). AI generates this perfectly given the
spec "GET /search?q=str&mode=Literal[...] returning SearchResponse with
latency_ms field". `app/main.py` is exactly that pattern — review the diff,
don't write it from scratch.

**Think hard yourself:** *what to measure*. Server-side latency vs wall-clock
vs client-side. P50 vs P95 vs P99. Cold vs warm. Single user vs concurrent.
These are *judgement* decisions: nếu rubric chỉ check P99, optimization sẽ
hướng vào tail latency, không phải mean. Đừng nhờ AI quyết định metric —
chỉ nhờ implement metric đã chọn.